# 17_generate_mel_nv_cue_qualitative_pdf

Generate qualitative GradCAM / Map Diff / FinerCAM comparison panels for the controlled binary MEL vs NV synthetic cue experiment.

This notebook uses the cued fixed center MEL/NV test set and compares:

1. Clean MEL/NV CE model
2. Cued MEL/NV CE model
3. Cued MEL/NV + HA model

The target/reference pair is class specific:

- GT = MEL: target MEL, reference NV
- GT = NV: target NV, reference MEL


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

QUAL_SEED = 42
CUE_SIZE = "small"      # small, big
TARGET_BLOCK_INDEX = -4

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"

FIXED_ROOT = HAM_ROOT / f"synthetic_cue_{CUE_SIZE}" / f"mel_nv_fixed_center_seed{QUAL_SEED}"
RANDOM_ROOT = HAM_ROOT / f"synthetic_cue_{CUE_SIZE}" / f"mel_nv_random_location_seed{QUAL_SEED}"

CLEAN_QUAL_CSV = FIXED_ROOT / "csv" / f"ham_mel_nv_clean_qualitative_10_seed{QUAL_SEED}.csv"

FIXED_CUE_QUAL_CSV = FIXED_ROOT / "csv" / f"ham_mel_nv_cue_fixed_qualitative_10_seed{QUAL_SEED}.csv"
RANDOM_CUE_QUAL_CSV = RANDOM_ROOT / "csv" / f"ham_mel_nv_cue_random_qualitative_10_seed{QUAL_SEED}.csv"

CHECKPOINT_ROOT = REPO_ROOT / "external" / f"checkpoints2_{CUE_SIZE}"

QUAL_ROOT = REPO_ROOT / "outputs" / f"overview_mel_nv_cue_{CUE_SIZE}_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}"
QUAL_ROOT.mkdir(parents=True, exist_ok=True)

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

print("CLEAN_QUAL_CSV:", CLEAN_QUAL_CSV)
print("FIXED_CUE_QUAL_CSV:", FIXED_CUE_QUAL_CSV)
print("RANDOM_CUE_QUAL_CSV:", RANDOM_CUE_QUAL_CSV)
print("QUAL_ROOT:", QUAL_ROOT)

for p in [CLEAN_QUAL_CSV, FIXED_CUE_QUAL_CSV, RANDOM_CUE_QUAL_CSV]:
    if not p.exists():
        print("[WARN] missing:", p)

CLEAN_QUAL_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean_qualitative_10_seed42.csv
FIXED_CUE_QUAL_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv
RANDOM_CUE_QUAL_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_random_location_seed42/csv/ham_mel_nv_cue_random_qualitative_10_seed42.csv
QUAL_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_mel_nv_cue_small_seed42_block-4


## 1. Create a small qualitative test CSV

This selects 5 MEL and 5 NV examples from the test split of the cued dataset.

For MEL images, the green cue is present. For NV images, no cue is present.

In [2]:
# df = pd.read_csv(CUE_CSV)

# required_cols = [
#     "image_id",
#     "gt_label",
#     "split",
#     "image_rel_path",
#     "mask_rel_path",
#     "cue_applied",
#     "cue_mask_rel_path",
# ]
# missing = [c for c in required_cols if c not in df.columns]
# if missing:
#     raise ValueError(f"Missing required columns in {CUE_CSV}: {missing}")

# test_df = df[df["split"] == "test"].copy()

# mel = test_df[test_df["gt_label"] == "MEL"].sample(n=5, random_state=QUAL_SEED)
# nv = test_df[test_df["gt_label"] == "NV"].sample(n=5, random_state=QUAL_SEED)

# qual_df = pd.concat([mel, nv], axis=0).copy()
# qual_df["order"] = qual_df["gt_label"].map({"MEL": 0, "NV": 1})
# qual_df = qual_df.sort_values(["order", "image_id"]).drop(columns=["order"])

# CUE_QUAL_CSV.parent.mkdir(parents=True, exist_ok=True)
# qual_df.to_csv(CUE_QUAL_CSV, index=False)

# print("Saved:", CUE_QUAL_CSV)
# print("Rows:", len(qual_df))
# print("Counts:")
# print(qual_df.groupby(["gt_label", "cue_applied"]).size())

# display(qual_df[[
#     "image_id",
#     "gt_label",
#     "split",
#     "image_rel_path",
#     "mask_rel_path",
#     "cue_applied",
#     "cue_mask_rel_path",
# ]])


## 2. Define experiments

Check that these checkpoint paths exist. If your output folder names differ slightly, adjust the paths here.

In [3]:
OVERVIEW_SCENARIOS = {
    "Clean CE on clean": {
        "checkpoint": CHECKPOINT_ROOT / "checkpoint-best-clean.pth",
        "checkpoint_model_type": "panderm",
        "csv": CLEAN_QUAL_CSV,
        "out_dir": QUAL_ROOT / "cam_clean_ce_on_clean",
    },
    "Clean CE on fixed cue": {
        "checkpoint": CHECKPOINT_ROOT / "checkpoint-best-clean.pth",
        "checkpoint_model_type": "panderm",
        "csv": FIXED_CUE_QUAL_CSV,
        "out_dir": QUAL_ROOT / "cam_clean_ce_on_fixed_cue",
    },
    "Cue CE on fixed cue": {
        "checkpoint": CHECKPOINT_ROOT / "checkpoint-best-cue.pth",
        "checkpoint_model_type": "panderm",
        "csv": FIXED_CUE_QUAL_CSV,
        "out_dir": QUAL_ROOT / "cam_cue_ce_on_fixed_cue",
    },
    "Cue HA on fixed cue": {
        "checkpoint": CHECKPOINT_ROOT / "checkpoint-best-cue-ha.pth",
        "checkpoint_model_type": "panderm",
        "csv": FIXED_CUE_QUAL_CSV,
        "out_dir": QUAL_ROOT / "cam_cue_ha_on_fixed_cue",
    },
    "Cue HA on random cue": {
        "checkpoint": CHECKPOINT_ROOT / "checkpoint-best-cue-ha.pth",
        "checkpoint_model_type": "panderm",
        "csv": RANDOM_CUE_QUAL_CSV,
        "out_dir": QUAL_ROOT / "cam_cue_ha_on_random_cue",
    },
}

for name, cfg in OVERVIEW_SCENARIOS.items():
    print(name)
    print("  checkpoint:", cfg["checkpoint"])
    print("  csv:", cfg["csv"])
    if not cfg["checkpoint"].exists():
        print("  [WARN] missing checkpoint")
    if not cfg["csv"].exists():
        print("  [WARN] missing CSV")

Clean CE on clean
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-clean.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean_qualitative_10_seed42.csv
Clean CE on fixed cue
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-clean.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv
Cue CE on fixed cue
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue.pth
  csv: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv
Cue HA on fixed cue
  checkpoint: /Users/choekyelnyungmartsang/Deve

## 3. Helper functions

In [4]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def generate_overview_cams(
    scenarios: dict,
    img_dir: Path,
    num_samples: int,
    dry_run: bool = False,
):
    for scenario_name, cfg in scenarios.items():
        out_dir = cfg["out_dir"]
        out_dir.mkdir(parents=True, exist_ok=True)

        panel_items = (
            "rgb_gt_mask,"
            "gradcam_a,"
            "gradcam_b,"
            "map_diff,"
            "finercam"
        )

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(cfg["csv"]),
            "--image_col", "image_rel_path",
            "--img_dir", str(img_dir),
            "--gt_col", "gt_label",
            "--checkpoint", str(cfg["checkpoint"]),
            "--checkpoint_model_type", cfg["checkpoint_model_type"],
            "--class_names", "MEL,NV",
            "--out_dir", str(out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--compare_mode", "gt_pair",
            "--A", "MEL",
            "--B", "NV",
            "--topk_compare", "1",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            "--target_block_index", str(TARGET_BLOCK_INDEX),
        ]

        print(f"\nRunning overview CAM generation: {scenario_name}")
        run_command(cmd, dry_run=dry_run)

## 4. Generate CAM panels

Set `dry_run=True` first if you only want to inspect the commands.

In [ ]:
generate_overview_cams(
    scenarios=OVERVIEW_SCENARIOS,
    img_dir=IMG_DIR,
    num_samples=10,
    dry_run=False,
)

## 5. Write PDF config

In [6]:
CONFIG_DIR = REPO_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

overview_pdf_config = [
    {"name": name, "folder": str(cfg["out_dir"].relative_to(REPO_ROOT))}
    for name, cfg in OVERVIEW_SCENARIOS.items()
]

OVERVIEW_JSON = CONFIG_DIR / f"overview_mel_nv_cue_{CUE_SIZE}_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}.json"
OVERVIEW_JSON.write_text(json.dumps(overview_pdf_config, indent=2))

print("Saved:", OVERVIEW_JSON)
print(json.dumps(overview_pdf_config, indent=2))

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/configs/overview_mel_nv_cue_small_seed42_block-4.json
[
  {
    "name": "Clean CE on clean",
    "folder": "outputs/overview_mel_nv_cue_small_seed42_block-4/cam_clean_ce_on_clean"
  },
  {
    "name": "Clean CE on fixed cue",
    "folder": "outputs/overview_mel_nv_cue_small_seed42_block-4/cam_clean_ce_on_fixed_cue"
  },
  {
    "name": "Cue CE on fixed cue",
    "folder": "outputs/overview_mel_nv_cue_small_seed42_block-4/cam_cue_ce_on_fixed_cue"
  },
  {
    "name": "Cue HA on fixed cue",
    "folder": "outputs/overview_mel_nv_cue_small_seed42_block-4/cam_cue_ha_on_fixed_cue"
  },
  {
    "name": "Cue HA on random cue",
    "folder": "outputs/overview_mel_nv_cue_small_seed42_block-4/cam_cue_ha_on_random_cue"
  }
]


## 6. Build comparison PDF

In [7]:
def build_qualitative_pdf(
    csv_path: Path,
    experiments_json_path: Path,
    out_pdf: Path,
    num_samples: int = 10,
    dry_run: bool = False,
):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "scripts.make_qualitative_comparison_pdf",
        "--csv", str(csv_path),
        "--image_col", "image_rel_path",
        "--gt_col", "gt_label",
        "--out_pdf", str(out_pdf),
        "--experiments_json_path", str(experiments_json_path),
        "--num_samples", str(num_samples),
        "--missing_policy", "placeholder",
    ]

    run_command(cmd, dry_run=dry_run)


build_qualitative_pdf(
    csv_path=FIXED_CUE_QUAL_CSV,
    experiments_json_path=OVERVIEW_JSON,
    out_pdf=QUAL_ROOT / f"overview_mel_nv_cue_{CUE_SIZE}_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}.pdf",
    num_samples=10,
    dry_run=False,
)


python -m scripts.make_qualitative_comparison_pdf --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv --image_col image_rel_path --gt_col gt_label --out_pdf /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_mel_nv_cue_small_seed42_block-4/overview_mel_nv_cue_small_seed42_block-4.pdf --experiments_json_path /Users/choekyelnyungmartsang/Developer/master-thesis/configs/overview_mel_nv_cue_small_seed42_block-4.json --num_samples 10 --missing_policy placeholder
Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/overview_mel_nv_cue_small_seed42_block-4/overview_mel_nv_cue_small_seed42_block-4.pdf
Pages written: 10


## Notes for interpretation

For MEL images, the green cue is present. The most important visual question is whether `Cue CE` and `Cue HA` place GradCAM / FinerCAM activation on that cue.

Expected pattern:

- `Clean CE`: no systematic focus on cue.
- `Cue CE`: likely strong focus on cue if the shortcut is learned.
- `Cue HA`: may also focus on cue because the cue is inside the lesion mask, so lesion HA does not penalize it.
